# Chapter 2 — The Minimal Agent Loop

*The spine that every later chapter extends.*

## Objective

Move from the hand-rolled loop of Chapter 1 to the `glassloop` primitives: `BaseAgent`, `Environment`, `run_loop` and `StepRecord`. Every step is inspectable; no intermediate state is hidden.

In [ ]:
from glassloop.core import (
    AgentState, BaseAgent, Finish, StepRecord, TaskSpec, ToolCall, run_loop,
)

## The agent

An agent subclasses `BaseAgent` and implements `propose_action(state) -> Action`. Default `update(state, action, observation)` appends the observation and advances the step counter.

In [ ]:
class GreedyAgent(BaseAgent):
    '''Proposes 'ping' tool calls until it has done `target` of them, then Finishes.'''
    def __init__(self, target: int) -> None:
        self.target = target
    def propose_action(self, state):
        if state.step >= self.target:
            return Finish(output={'pings': state.step})
        return ToolCall(tool_name='ping', arguments={})

## The environment

An environment is anything with a `step(action) -> dict` method. Production agents run actions through a `GovernedToolExecutor` (Chapter 6); here we use a tiny mock.

In [ ]:
class PingEnv:
    def step(self, action):
        return {'pong': True}

## Run the loop

`run_loop` is a generator. Each yielded `StepRecord` carries the state before, the action, the observation and the state after.

In [ ]:
task = TaskSpec(goal='do three pings')
state = AgentState(task=task)
records = list(run_loop(GreedyAgent(target=3), PingEnv(), state, max_steps=10))
print(f'{len(records)} step records yielded')
print(f'final status: {records[-1].state_after.status}')
print(f'final output: {records[-1].state_after.final_output}')

Each record is a self-contained snapshot of a step:

In [ ]:
for r in records:
    print(f'step {r.step}: action={r.action.kind:<10} obs={r.observation}')

## What the loop does and does not do

- It terminates on `Finish`, on `Escalate`, on `state.status != 'running'`, or on `max_steps`.
- It does **not** call any LLM. The agent is a plain Python object.
- Token, time and tool-call budgets are added in Chapter 7. Governance gates and audit are added in Chapter 12. Both extend this loop without rewriting it.

## Anti-patterns flagged here

- Loops that hide intermediate state.
- Mixing the loop driver with the policy.
- Treating the LLM call as the loop.

In [ ]:
# Self-check
assert all(isinstance(r, StepRecord) for r in records)
assert records[-1].state_after.status == 'done'
assert records[-1].state_after.final_output == {'pings': 3}
print('OK')